# Task 6 — Generate Analysis and a Chart

Revenue by category and by store, top 5 products by revenue, and a return rate -- built directly on the `fact_transaction` / `fact_transaction_line` model from Task 4, joined against the cleaned `dim_product` attributes.

**Required caveat, carried forward from Task 5:** `ST99` (unmapped store) accounts for ~33% of revenue. Any store-level breakdown below must show this honestly, not hide it.

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt

conn = sqlite3.connect("../output/practice.db")

# one row per line item, joined to store and cleaned product attributes
lines = pd.read_sql("""
    SELECT ftl.line_id, ftl.transaction_id, ftl.product_sku, ftl.qty, ftl.unit_price,
           ft.store_id, dp.category, dp.product_name
    FROM fact_transaction_line ftl
    JOIN fact_transaction ft ON ftl.transaction_id = ft.transaction_id
    LEFT JOIN dim_product dp ON ftl.product_sku = dp.product_sku
""", conn)

lines["revenue"] = lines["qty"] * lines["unit_price"]

# check: a handful of nulls are expected here -- Task 1 found 3 rows with
# unresolvable qty/unit_price, which correctly propagate to a null revenue
# rather than being silently treated as zero
print("lines:", len(lines))
print("lines with null revenue (missing qty or price):", lines["revenue"].isna().sum())

## Revenue by category

In [ ]:
revenue_by_category = lines.groupby("category")["revenue"].sum().round(2).sort_values(ascending=False)

# check: 4 categories, matching Task 1's case-folded count -- not 8
print("distinct categories in output:", revenue_by_category.shape[0])
revenue_by_category

## Revenue by store

Reported honestly, including the unmapped store -- not folded into another store or dropped.

In [ ]:
revenue_by_store = lines.groupby("store_id")["revenue"].sum().round(2).sort_values(ascending=False)
print(revenue_by_store)

# check: quantify the ST99 caveat directly, don't just gesture at it
st99_share = revenue_by_store.get("ST99", 0) / revenue_by_store.sum() * 100
print(f"\nST99 (unmapped store) share of total revenue: {st99_share:.1f}%")

## Top 5 products by revenue

In [ ]:
top_products = lines.groupby("product_name")["revenue"].sum().round(2).sort_values(ascending=False).head(5)

# check: labels should read as clean, title-cased product names --
# a lowercase 'ceramic mug' here would mean the dim_product cleanup regressed
print(top_products)

## Return rate

In [ ]:
return_lines = (lines["qty"] < 0).sum()
total_lines = len(lines)
return_rate_by_count = return_lines / total_lines * 100

return_revenue = lines.loc[lines["qty"] < 0, "revenue"].sum()
gross_revenue = lines.loc[lines["qty"] > 0, "revenue"].sum()
return_rate_by_dollar = abs(return_revenue) / gross_revenue * 100

# check: report both views -- rate by line count and rate by dollar value can
# tell different stories (a few high-value returns vs many low-value ones)
print(f"return lines: {return_lines} / {total_lines} ({return_rate_by_count:.1f}%)")
print(f"return $ as share of gross revenue: {return_rate_by_dollar:.1f}%")

## Chart: revenue by store

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

# color the unmapped store differently so the caveat is visible in the
# chart itself, not just in a footnote someone might skip
colors = ["#d62728" if store == "ST99" else "#4c72b0" for store in revenue_by_store.index]

ax.bar(revenue_by_store.index, revenue_by_store.values, color=colors)
ax.set_title("Revenue by Store")
ax.set_xlabel("Store")
ax.set_ylabel("Revenue ($)")

for i, v in enumerate(revenue_by_store.values):
    ax.text(i, v + 20, f"${v:,.0f}", ha="center", fontsize=9)

ax.text(0.5, -0.22, "Red = ST99, an unmapped store code not present in the store reference file (see Task 5)",
        transform=ax.transAxes, ha="center", fontsize=8, color="#666666")

plt.tight_layout()
plt.savefig("../output/revenue_by_store.png", dpi=150, bbox_inches="tight")
plt.show()

## Summary

| Metric | Result |
|---|---|
| Top category by revenue | Home (~$1,902) |
| Top store by revenue | `ST99` (~$1,956, ~33% of total) -- **unmapped, see Task 5** |
| Top product by revenue | Notebook - Recycled (~$900) |
| Return rate | 1.5% of lines, 0.6% of gross revenue |

Exact figures will vary slightly depending on the random seed used to generate the practice data, but the shape of the findings -- and the requirement to caveat `ST99` in any store-level view -- holds regardless.

Ready for Task 7: turn this into a 3-5 sentence, plain-language narrative for a non-technical stakeholder.

In [ ]:
conn.close()